# EasyRAG Fusion Rerank And Top-K

## What you will do

- inspect candidate lists before fusion
- compare weighted merge and reciprocal-rank fusion on the same records
- apply a deterministic reranker and a final top-k cut

## Why this matters

Retrieval is not a single ranked list inside EasyRAG. Multiple candidate sources can contribute evidence, and the system needs a visible way to combine them before answer generation.


## Source code anchors

- `easyrag.rag.retrieval.fusion.merge_ranked_records`
- `easyrag.rag.retrieval.fusion.rrf_fuse`
- `easyrag.rag.retrieval.fusion.combine_mode_results`
- `tests.test_rag_retriever._stub_reranker`
- `easyrag.rag.types.QueryParam`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.retrieval.fusion import combine_mode_results, merge_ranked_records, rrf_fuse


## Deterministic path

We first inspect fusion outside the full retriever so the score movement stays obvious. After that, we will run a small `EasyRAG` query to see the same ideas in a real result object.


In [ ]:
query_text = "How do rewrite and rerank improve retrieval?"
naive_candidates = [
    {"id": "doc::arch::0", "score": 3.2, "text": "query rewrite expands retrieval coverage", "source": "naive"},
    {"id": "doc::faq::0", "score": 2.4, "text": "metadata filters narrow the final evidence set", "source": "naive"},
]
local_candidates = [
    {"id": "doc::arch::0", "score": 2.1, "text": "query rewrite expands retrieval coverage", "source": "local"},
    {"id": "doc::policy::0", "score": 1.8, "text": "rerank lifts grounded chunks before answering", "source": "local"},
]
global_candidates = [
    {"id": "doc::policy::0", "score": 2.6, "text": "rerank lifts grounded chunks before answering", "source": "global"},
    {"id": "doc::graph::0", "score": 1.9, "text": "graph relations add extra retrieval context", "source": "global"},
]

print("Naive candidates")
pprint(naive_candidates)
print("\nLocal candidates")
pprint(local_candidates)
print("\nGlobal candidates")
pprint(global_candidates)

weighted_merge = merge_ranked_records([(1.0, naive_candidates), (0.7, local_candidates), (0.8, global_candidates)])
rrf_merge = rrf_fuse([naive_candidates, local_candidates, global_candidates])
combined_rrf = combine_mode_results(
    QueryParam(retrieval_fusion="rrf"),
    (1.0, naive_candidates),
    (0.7, local_candidates),
    (0.8, global_candidates),
)

print("\nWeighted merge")
pprint(weighted_merge)
print("\nRRF merge")
pprint(rrf_merge)
print("\ncombine_mode_results(QueryParam(retrieval_fusion='rrf'), ...) output")
pprint(combined_rrf)


## Output inspection

The fused list is still not the final answer budget. A reranker can change the order again, and a top-k budget still trims the tail before hydration or generation.


In [ ]:
reranked = _stub_reranker(query_text, combined_rrf)
top_k_records = reranked[:2]

print("Reranked fused records")
pprint(reranked)
print("\nTop-2 after rerank")
pprint(top_k_records)


In [ ]:
fusion_tmp = tempfile.TemporaryDirectory()
fusion_rag = EasyRAG(
    working_dir=fusion_tmp.name,
    workspace="fusion-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    reranker_func=_stub_reranker,
)
run_sync(fusion_rag.initialize_storages())
run_sync(
    fusion_rag.ainsert(
        [
            "# Architecture\nQuery rewrite expands retrieval coverage and keeps the candidate pool inspectable.\n",
            "# Policy\nRerank lifts grounded chunks before answer synthesis.\n",
            "# Graph\nGraph relations add extra retrieval context when local evidence is sparse.\n",
        ],
        ids=["doc::arch", "doc::policy", "doc::graph"],
        file_paths=["docs/architecture.md", "docs/policy.md", "docs/graph.md"],
    )
)

without_rerank = run_sync(
    fusion_rag.aquery(
        query_text,
        QueryParam(mode="hybrid", enable_rerank=False, chunk_top_k=3),
    )
)
with_rerank = run_sync(
    fusion_rag.aquery(
        query_text,
        QueryParam(mode="hybrid", enable_rerank=True, chunk_top_k=2),
    )
)

print("Without rerank")
_print_json(without_rerank.metadata)
print("\nWith rerank and tighter top-k")
_print_json(with_rerank.metadata)
print("\nFinal citations after rerank")
_print_json(with_rerank.citations)


## Provider-backed path

The optional path uses the repo's default providers when available. The output is still interpreted the same way: candidate lists are combined, optionally reranked, then trimmed to the answer budget.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-fusion-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nQuery rewrite and rerank are part of the retrieval stack.\n",
                    "# Evidence\nTop-k trimming keeps answer prompts compact.\n",
                ],
                ids=["doc::retrieval", "doc::evidence"],
                file_paths=["docs/retrieval.md", "docs/evidence.md"],
            )
        )
        provider_result = run_sync(provider_rag.aquery(query_text, QueryParam(mode="hybrid", enable_rerank=True, chunk_top_k=2)))
        _print_json(provider_result.metadata)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Fusion changes how evidence from different retrieval perspectives competes. Rerank then applies a second ordering criterion on the fused list. Finally, top-k enforces the budget that the rest of the pipeline must live with. Keeping those three steps separate makes ranking behavior much easier to diagnose.


In [ ]:
run_sync(fusion_rag.finalize_storages())
fusion_tmp.cleanup()
print("Cleaned up the deterministic fusion workspace.")


## Next steps

- Continue with [05_01_query_result_to_answer.ipynb](../05_generation/05_01_query_result_to_answer.ipynb) to see how the final retrieval result becomes answerable evidence.
- Read [04-retrieval-overview.md](../../docs/04-retrieval-overview.md) for the full retrieval-stage narrative, including fusion, rerank, and fallback behavior.
- Compare this notebook with [04_04_hybrid_metadata_filter_and_modes.ipynb](04_04_hybrid_metadata_filter_and_modes.ipynb) if you want to step back to mode-level retrieval choices.
